# EDA Notebook for Dataset v2

Full statistical overview, reconciliation cross-check, and data-quality drilldown for `data/v2/`.

**Architecture (002-eda-v2-dataset-notebook):**
- Helpers: [`src/eda/dataset_v2.py`](../src/eda/dataset_v2.py) — streaming counts, reservoir sampling, reconciliation, tag tallies
- Notebook: thin orchestration (config + display only)

**Run order:** Environment → Config → Preflight → Documents → Edges → Text/Structure → Validity → Authority → Reconciliation → Quality

Large files (`chunks.jsonl`, `provisions.jsonl`) are **streamed / reservoir-sampled only** — never fully loaded into memory (FR-002, SC-003).


## 1. Environment setup


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    # Useful if the notebook is launched from notebooks/.
    PROJECT_ROOT = Path.cwd().parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from eda.dataset_v2 import resolve_project_root

PROJECT_ROOT = resolve_project_root(Path.cwd())
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Project root:", PROJECT_ROOT)
print("src on path:", SRC_DIR.exists())


## 2. Config

Configure paths and sampling near the top (FR-012). Do not hardcode these deeper in the notebook.


In [ ]:
from eda.dataset_v2 import (
    PreflightResult,
    coerce_category,
    iter_jsonl,
    lookup_by_key,
    preflight,
    reconcile,
    reservoir_sample,
    stream_count,
    tally_tags,
    vocab_coverage,
)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display

# --- user-configurable parameters (FR-012) ---
DATASET_ROOT = PROJECT_ROOT / "data" / "v2"
UNTRACKED_ROOT = PROJECT_ROOT / "data" / "untracked_data"
SAMPLE_SIZE = 5000
SAMPLE_SEED = 42
STREAM_SIZE_THRESHOLD_BYTES = 50 * 1024 * 1024

# Convenience paths
PATHS = {
    "documents": DATASET_ROOT / "documents.jsonl",
    "documents_quarantine": DATASET_ROOT / "documents_quarantine.jsonl",
    "edges": DATASET_ROOT / "edges.jsonl",
    "edges_quarantine": DATASET_ROOT / "edges_quarantine.jsonl",
    "external_stubs": DATASET_ROOT / "external_stubs.jsonl",
    "provisions": DATASET_ROOT / "provisions.jsonl",
    "chunks": DATASET_ROOT / "chunks.jsonl",
    "text_provenance": DATASET_ROOT / "text_provenance.jsonl",
    "validity_timeline": DATASET_ROOT / "validity_timeline.jsonl",
    "authority_index": DATASET_ROOT / "authority_index.jsonl",
    "reconciliation_report": DATASET_ROOT / "reconciliation_report.md",
    "vocabularies": DATASET_ROOT / "vocabularies",
    "metadata_raw": UNTRACKED_ROOT / "metadata.jsonl",
    "relationships_raw": UNTRACKED_ROOT / "relationships.jsonl",
}

print("DATASET_ROOT:", DATASET_ROOT)
print("SAMPLE_SIZE:", SAMPLE_SIZE, "| SAMPLE_SEED:", SAMPLE_SEED)
print("STREAM_SIZE_THRESHOLD_BYTES:", STREAM_SIZE_THRESHOLD_BYTES)


def counter_to_df(counter: Counter, name: str = "value") -> pd.DataFrame:
    if not counter:
        return pd.DataFrame(columns=[name, "count", "pct"])
    total = sum(counter.values()) or 1
    rows = sorted(counter.items(), key=lambda kv: (-kv[1], str(kv[0])))
    return pd.DataFrame(
        {
            name: [k for k, _ in rows],
            "count": [v for _, v in rows],
            "pct": [round(100.0 * v / total, 2) for _, v in rows],
        }
    )


def show_distribution(counter: Counter, title: str, name: str = "value", top_n: int = 30, plot: bool = True):
    """Table + optional horizontal bar chart (R5: table is source of truth)."""
    df = counter_to_df(counter, name=name)
    print(f"\n=== {title} ===")
    if df.empty:
        print("(empty)")
        return df
    display(df.head(top_n) if len(df) > top_n else df)
    if plot and not df.empty and len(df) <= 40:
        plot_df = df.head(min(top_n, 25)).iloc[::-1]
        fig, ax = plt.subplots(figsize=(8, max(2.5, 0.28 * len(plot_df))))
        ax.barh(plot_df[name].astype(str), plot_df["count"], color="steelblue")
        ax.set_title(title)
        ax.set_xlabel("count")
        plt.tight_layout()
        plt.show()
    elif plot and len(df) > 40:
        print(f"(plot skipped: {len(df)} categories; showing table only)")
    return df


def file_ready(key: str) -> bool:
    path = PATHS[key]
    ok = path.is_file() if path.suffix else path.is_dir()
    if not ok:
        print(f"[SKIP] missing artifact: {path}")
    return ok


## 3. Preflight (FR-001)


In [ ]:
pf: PreflightResult = preflight(PROJECT_ROOT)

rows = []
for name, ok in sorted(pf.by_artifact.items()):
    rows.append({"artifact": name, "present": ok})
pf_df = pd.DataFrame(rows)
display(pf_df)

missing_names = [k for k, v in pf.by_artifact.items() if not v]
if missing_names:
    print("Missing artifacts:")
    for name in missing_names:
        print(" -", name)
else:
    print("All expected artifacts present.")

print(f"present paths: {len(pf.present)} | missing paths: {len(pf.missing)}")


## 4. Documents overview (FR-003)

Distributions for authority rank, document type, validity, currency, controlled-vocab facets, and issue year.


In [ ]:
if file_ready("documents"):
    doc_fields = [
        "legal_authority_rank",
        "loai_van_ban",
        "validity_group",
        "currency_hint",
        "scope.code",
        "legal_field.code",
        "issuing_authority.code",
        "sector.code",
        "issue_year",
    ]
    doc_stats = stream_count(PATHS["documents"], doc_fields)
    print(f"documents total_rows={doc_stats.total_rows:,} | malformed_lines={doc_stats.malformed_lines}")

    for field in doc_fields:
        show_distribution(
            doc_stats.field_counters[field],
            title=f"documents · {field}",
            name=field.split(".")[-1],
            top_n=25,
            plot=field != "legal_field.code",  # high-ish cardinality → table first
        )
else:
    doc_stats = None
    print("Documents section skipped.")


## 5. Edges overview (FR-004)


In [ ]:
if file_ready("edges"):
    edge_fields = [
        "rel_canonical",
        "rel_group",
        "direction_verified",
        "external_target",
    ]
    edge_stats = stream_count(PATHS["edges"], edge_fields)
    print(f"edges total_rows={edge_stats.total_rows:,} | malformed_lines={edge_stats.malformed_lines}")

    for field in edge_fields:
        show_distribution(
            edge_stats.field_counters[field],
            title=f"edges · {field}",
            name=field,
            top_n=30,
        )

    dv = edge_stats.field_counters["direction_verified"]
    et = edge_stats.field_counters["external_target"]
    n = edge_stats.total_rows or 1
    print(
        f"direction_verified true share: {100.0 * dv.get('true', 0) / n:.2f}% | "
        f"false: {100.0 * dv.get('false', 0) / n:.2f}%"
    )
    print(
        f"external_target true share: {100.0 * et.get('true', 0) / n:.2f}% | "
        f"false: {100.0 * et.get('false', 0) / n:.2f}%"
    )
else:
    edge_stats = None
    print("Edges section skipped.")


## 6. Text / structure layer (FR-002, FR-005)

`text_provenance` is fully aggregated. `provisions.jsonl` / `chunks.jsonl` use **streaming aggregates** and **seeded reservoir sampling** for record-level inspection.


In [ ]:
if file_ready("text_provenance"):
    prov_fields = ["text_status", "content_row_count", "structuring_status"]
    prov_stats = stream_count(PATHS["text_provenance"], prov_fields)
    print(
        f"text_provenance total_rows={prov_stats.total_rows:,} | "
        f"malformed_lines={prov_stats.malformed_lines}"
    )
    for field in prov_fields:
        show_distribution(
            prov_stats.field_counters[field],
            title=f"text_provenance · {field}",
            name=field,
        )
else:
    prov_stats = None
    print("text_provenance section skipped.")


In [ ]:
# Provisions: stream aggregates only (never full load)
if file_ready("provisions"):
        prov_total = 0
    chunks_per_provision = []
    unit_types = Counter()
    docs_seen = Counter()  # id_str -> provision count
    malformed = [0]

    for row in iter_jsonl(PATHS["provisions"], skip_counter=malformed):
        prov_total += 1
        unit_types[coerce_category(row.get("unit_type"))] += 1
        id_str = row.get("id_str")
        if id_str is not None:
            docs_seen[str(id_str)] += 1
        cc = row.get("chunk_count")
        if isinstance(cc, (int, float)):
            chunks_per_provision.append(int(cc))

    print(f"provisions total_rows={prov_total:,} | malformed_lines={malformed[0]}")
    print(f"distinct documents with provisions: {len(docs_seen):,}")
    show_distribution(unit_types, title="provisions · unit_type", name="unit_type")

    if chunks_per_provision:
        arr = np.asarray(chunks_per_provision, dtype=np.float64)
        summary = {
            "mean": float(arr.mean()),
            "median": float(np.median(arr)),
            "p90": float(np.percentile(arr, 90)),
            "p99": float(np.percentile(arr, 99)),
            "max": float(arr.max()),
        }
        print("chunks_per_provision summary:")
        display(pd.DataFrame([summary]))

    if docs_seen:
        ppd = np.asarray(list(docs_seen.values()), dtype=np.float64)
        print("provisions_per_document summary:")
        display(
            pd.DataFrame(
                [
                    {
                        "mean": float(ppd.mean()),
                        "median": float(np.median(ppd)),
                        "p90": float(np.percentile(ppd, 90)),
                        "p99": float(np.percentile(ppd, 99)),
                        "max": float(ppd.max()),
                    }
                ]
            )
        )
else:
    print("Provisions section skipped.")


In [ ]:
# Chunks: stream length aggregates + reservoir sample for inspection
if file_ready("chunks"):
        chunk_total = 0
    char_counts = []
    malformed = [0]
    for row in iter_jsonl(PATHS["chunks"], skip_counter=malformed):
        chunk_total += 1
        n = row.get("chunk_char_count")
        if n is None and isinstance(row.get("chunk_text"), str):
            n = len(row["chunk_text"])
        if isinstance(n, (int, float)):
            char_counts.append(int(n))

    print(f"chunks total_rows={chunk_total:,} | malformed_lines={malformed[0]}")
    if char_counts:
        arr = np.asarray(char_counts, dtype=np.float64)
        print("chunk_char_count full-corpus summary (streamed):")
        display(
            pd.DataFrame(
                [
                    {
                        "mean": float(arr.mean()),
                        "median": float(np.median(arr)),
                        "p90": float(np.percentile(arr, 90)),
                        "p99": float(np.percentile(arr, 99)),
                        "max": float(arr.max()),
                    }
                ]
            )
        )

    sample = reservoir_sample(PATHS["chunks"], sample_size=SAMPLE_SIZE, seed=SAMPLE_SEED)
    print(
        f"Reservoir sample method=Algorithm R | seed={sample.seed} | "
        f"sample_size={sample.sample_size} | rows_seen={sample.rows_seen:,} | "
        f"returned={len(sample.sample):,}"
    )

    sample_lens = []
    for row in sample.sample:
        n = row.get("chunk_char_count")
        if n is None and isinstance(row.get("chunk_text"), str):
            n = len(row["chunk_text"])
        if isinstance(n, (int, float)):
            sample_lens.append(int(n))
    if sample_lens:
        sarr = np.asarray(sample_lens, dtype=np.float64)
        print("sampled chunk length distribution:")
        display(
            pd.DataFrame(
                [
                    {
                        "mean": float(sarr.mean()),
                        "median": float(np.median(sarr)),
                        "p90": float(np.percentile(sarr, 90)),
                        "p99": float(np.percentile(sarr, 99)),
                        "max": float(sarr.max()),
                    }
                ]
            )
        )
        fig, ax = plt.subplots(figsize=(8, 3.5))
        ax.hist(sarr, bins=40, color="steelblue", edgecolor="white")
        ax.set_title(f"Sampled chunk_char_count (n={len(sarr)}, seed={SAMPLE_SEED})")
        ax.set_xlabel("characters")
        ax.set_ylabel("count")
        plt.tight_layout()
        plt.show()

    # Resolve a few sample rows back to id_str / citation (Constitution I)
    print("Sample rows resolved to document identity (first 5):")
    preview = []
    for row in sample.sample[:5]:
        id_str = row.get("id_str")
        parent = row.get("parent_unit_id")
        citation = None
        if id_str and PATHS["documents"].is_file():
            doc = lookup_by_key(PATHS["documents"], "id_str", str(id_str))
            if doc:
                citation = doc.get("citation_label") or doc.get("title")
        preview.append(
            {
                "chunk_id": row.get("chunk_id"),
                "parent_unit_id": parent,
                "id_str": id_str,
                "citation_label": citation,
                "chunk_char_count": row.get("chunk_char_count"),
            }
        )
    display(pd.DataFrame(preview))
else:
    print("Chunks section skipped.")


## 7. Validity timeline (FR-006)

`direction_verified = false` is **pending sign-off / not production-ready** (Dataset_SPEC_v2 §8.2).


In [ ]:
if file_ready("validity_timeline"):
    val_fields = ["event_type", "direction_verified"]
    val_stats = stream_count(PATHS["validity_timeline"], val_fields)
    print(
        f"validity_timeline total_events={val_stats.total_rows:,} | "
        f"malformed_lines={val_stats.malformed_lines}"
    )
    show_distribution(val_stats.field_counters["event_type"], title="validity · event_type", name="event_type")
    show_distribution(
        val_stats.field_counters["direction_verified"],
        title="validity · direction_verified",
        name="direction_verified",
    )

    dv = val_stats.field_counters["direction_verified"]
    n = val_stats.total_rows or 1
    verified = dv.get("true", 0)
    unverified = dv.get("false", 0)
    print(
        f"direction_verified split: verified={verified:,} ({100.0 * verified / n:.2f}%) | "
        f"unverified={unverified:,} ({100.0 * unverified / n:.2f}%)"
    )
    print(
        "NOTE: direction_verified=false events are PENDING SIGN-OFF and MUST NOT be "
        "treated as production-ready (reconciliation report P1 / Dataset_SPEC_v2 §8.2)."
    )
else:
    val_stats = None
    print("Validity section skipped.")


## 8. Authority index (FR-007)


In [ ]:
if file_ready("authority_index"):
    auth_rows = []
        for row in iter_jsonl(PATHS["authority_index"]):
        auth_rows.append(
            {
                "loai_van_ban": row.get("loai_van_ban"),
                "legal_authority_rank": row.get("legal_authority_rank"),
                "rank_label": row.get("rank_label"),
            }
        )
    auth_df = pd.DataFrame(auth_rows).sort_values(
        by=["legal_authority_rank", "loai_van_ban"], kind="stable"
    )
    print(f"authority_index rows: {len(auth_df)}")
    display(auth_df)

    # Cross-check vs documents
    if file_ready("documents"):
        doc_stats_local = globals().get("doc_stats")
        if doc_stats_local is not None and "loai_van_ban" in doc_stats_local.field_counters:
            doc_types = set(doc_stats_local.field_counters["loai_van_ban"].keys())
        else:
            sc = stream_count(PATHS["documents"], ["loai_van_ban"])
            doc_types = set(sc.field_counters["loai_van_ban"].keys())
        ranked = {str(r["loai_van_ban"]) for _, r in auth_df.iterrows()}
        unranked = sorted(t for t in doc_types if t not in ranked and t != "(missing)")
        rank_map = {
            str(r["loai_van_ban"]): r["legal_authority_rank"] for _, r in auth_df.iterrows()
        }
        fallback_99 = sorted(
            t for t in doc_types if rank_map.get(t) in (99, "99") or t in unranked
        )
        print(f"distinct loai_van_ban in documents: {len(doc_types)}")
        print(f"not present in authority_index: {unranked[:50]}")
        print(f"unranked / rank 99 fallbacks (sample): {fallback_99[:50]}")
else:
    print("Authority index section skipped.")


## 9. Reconciliation (FR-008, US2)

Independently recompute `raw == final + quarantine` for documents and edges, then compare to `reconciliation_report.md`.


In [ ]:
recon_rows = []

if (
    file_ready("metadata_raw")
    and file_ready("documents")
    and file_ready("documents_quarantine")
):
    doc_check = reconcile(
        PATHS["metadata_raw"],
        PATHS["documents"],
        PATHS["documents_quarantine"],
        PATHS["reconciliation_report"],
        "documents",
    )
    recon_rows.append(doc_check)
else:
    doc_check = None
    print("[SKIP] documents reconciliation — missing raw/final/quarantine inputs")

if (
    file_ready("relationships_raw")
    and file_ready("edges")
    and file_ready("edges_quarantine")
):
    edge_check = reconcile(
        PATHS["relationships_raw"],
        PATHS["edges"],
        PATHS["edges_quarantine"],
        PATHS["reconciliation_report"],
        "edges",
    )
    recon_rows.append(edge_check)
else:
    edge_check = None
    print("[SKIP] edges reconciliation — missing raw/final/quarantine inputs")

if recon_rows:
    display(
        pd.DataFrame(
            [
                {
                    "label": c.label,
                    "raw": c.raw_count,
                    "final": c.final_count,
                    "quarantine": c.quarantine_count,
                    "identity_holds": c.identity_holds,
                    "report_raw": c.report_raw,
                    "report_final": c.report_final,
                    "report_quarantine": c.report_quarantine,
                    "matches_report": c.matches_report,
                    "status": (
                        "PASS"
                        if c.identity_holds and c.matches_report
                        else ("IDENTITY_OK_REPORT_MISMATCH" if c.identity_holds else "FAIL")
                    ),
                }
                for c in recon_rows
            ]
        )
    )
    for c in recon_rows:
        if not c.identity_holds:
            print(
                f"DELTA {c.label}: recomputed raw={c.raw_count} final={c.final_count} "
                f"quarantine={c.quarantine_count} (sum final+q={c.final_count + c.quarantine_count})"
            )
        if not c.matches_report:
            print(
                f"REPORT MISMATCH {c.label}: report=({c.report_raw}, {c.report_final}, "
                f"{c.report_quarantine}) vs recomputed=({c.raw_count}, {c.final_count}, {c.quarantine_count})"
            )

# Cross-link validity verified/unverified narrative (T029)
val_stats_local = globals().get("val_stats")
if val_stats_local is not None:
    dv = val_stats_local.field_counters["direction_verified"]
    print(
        "\nValidity direction sign-off (cross-link): "
        f"verified={dv.get('true', 0):,} | unverified={dv.get('false', 0):,} "
        "→ unverified majority is pending sign-off, not production-ready."
    )


## 10. Quality drilldown (FR-009–FR-011, US3)

Quarantine multi-tag reasons, text_status / html flags, external stubs, controlled-vocabulary coverage.


In [ ]:
# Quarantine reasons — rows authoritative, tags independent (Dataset_SPEC_v2 §9)
if file_ready("documents_quarantine"):
    doc_tags = tally_tags(PATHS["documents_quarantine"], "exclusion_reasons")
    print(
        f"documents_quarantine row_count={doc_tags.row_count:,} | "
        f"distinct reason tags={len(doc_tags.tag_counts)}"
    )
    show_distribution(doc_tags.tag_counts, title="documents_quarantine · exclusion_reasons", name="reason")
else:
    print("documents_quarantine quality section skipped.")

if file_ready("edges_quarantine"):
    # Prefer edge_quality_flags; also surface exclusion_reasons if present
    edge_flag_tags = tally_tags(PATHS["edges_quarantine"], "edge_quality_flags")
    edge_reason_tags = tally_tags(PATHS["edges_quarantine"], "exclusion_reasons")
    print(
        f"edges_quarantine row_count={edge_flag_tags.row_count:,} | "
        f"edge_quality_flags tags={len(edge_flag_tags.tag_counts)} | "
        f"exclusion_reasons tags={len(edge_reason_tags.tag_counts)}"
    )
    if edge_flag_tags.tag_counts:
        show_distribution(
            edge_flag_tags.tag_counts,
            title="edges_quarantine · edge_quality_flags",
            name="flag",
        )
    if edge_reason_tags.tag_counts:
        show_distribution(
            edge_reason_tags.tag_counts,
            title="edges_quarantine · exclusion_reasons",
            name="reason",
        )
else:
    print("edges_quarantine quality section skipped.")


In [ ]:
# text_status + html_quality_flags
if file_ready("text_provenance"):
        status_counts = Counter()
    html_flag_counts = Counter()
    n = 0
    for row in iter_jsonl(PATHS["text_provenance"]):
        n += 1
        status_counts[coerce_category(row.get("text_status"))] += 1
        flags = row.get("html_quality_flags") or []
        if isinstance(flags, str):
            flags = [flags]
        if not flags:
            html_flag_counts["(none)"] += 1
        else:
            for f in flags:
                html_flag_counts[str(f)] += 1

    print(f"text_provenance quality rows={n:,}")
    show_distribution(status_counts, title="quality · text_status", name="text_status")
    # tag-style for html flags: show tag table + note row denominator
    print(f"html_quality_flags tag tallies (row denominator={n:,}; multi-tag rows contribute to multiple tags)")
    show_distribution(html_flag_counts, title="quality · html_quality_flags", name="flag")
else:
    print("text_provenance quality flags skipped.")


In [ ]:
# External stubs (FR-010)
if file_ready("external_stubs"):
        stub_ids = set()
    citation_safe_counts = Counter()
    ref_counts = Counter()
    ref_values = []
    for row in iter_jsonl(PATHS["external_stubs"]):
        sid = row.get("id_str")
        if sid is not None:
            stub_ids.add(str(sid))
        citation_safe_counts[coerce_category(row.get("citation_safe"))] += 1
        rc = row.get("referenced_by_edge_count")
        ref_counts[coerce_category(rc)] += 1
        if isinstance(rc, (int, float)):
            ref_values.append(int(rc))

    print(f"external_stubs distinct id_str: {len(stub_ids):,}")
    show_distribution(citation_safe_counts, title="external_stubs · citation_safe", name="citation_safe")
    if ref_values:
        arr = np.asarray(ref_values, dtype=np.float64)
        print("referenced_by_edge_count summary:")
        display(
            pd.DataFrame(
                [
                    {
                        "mean": float(arr.mean()),
                        "median": float(np.median(arr)),
                        "p90": float(np.percentile(arr, 90)),
                        "p99": float(np.percentile(arr, 99)),
                        "max": float(arr.max()),
                        "sum": int(arr.sum()),
                    }
                ]
            )
        )
    # compact distribution of reference counts (binned-ish via value_counts table, not raw ids)
    show_distribution(ref_counts, title="external_stubs · referenced_by_edge_count", name="referenced_by_edge_count", top_n=40, plot=False)
else:
    print("external_stubs section skipped.")


In [ ]:
# Controlled vocabulary coverage (FR-011 / SC-005)
facets = ["issuing_authority", "legal_field", "sector", "scope"]
if file_ready("documents"):
    cov_rows = []
    for facet in facets:
        cov = vocab_coverage(PATHS["documents"], facet)
        cov_rows.append(
            {
                "facet": cov.facet,
                "total": cov.total,
                "unmapped_or_missing": cov.unmapped_or_missing,
                "pct_unmapped_or_missing": round(cov.pct_unmapped_or_missing, 4),
            }
        )
    cov_df = pd.DataFrame(cov_rows)
    display(cov_df)

    # Optional: surface vocab JSON presence
    vocab_dir = PATHS["vocabularies"]
    if vocab_dir.is_dir():
        for facet in facets:
            vp = vocab_dir / f"{facet}.json"
            print(f"vocab file {vp.name}: {'present' if vp.is_file() else 'MISSING'}")
    else:
        print(
            "vocabularies/ directory missing — UNMAPPED/MISSING % above still computed "
            "from documents.jsonl codes (edge case: unmapped-but-referenced codes)."
        )
else:
    print("vocab coverage section skipped.")


## Done

Checklist:
- Preflight listed present/missing artifacts without crashing
- Large files streamed / reservoir-sampled (seed + sample_size + rows_seen printed)
- Reconciliation PASS/FAIL with recomputed vs report triples
- Quality: quarantine tags, text flags, stubs, vocab UNMAPPED/MISSING %
- Sampled chunks resolved to `id_str` / citation where possible
- `direction_verified=false` labeled pending sign-off
